# Logistic regression — from scratch

*Companion to **Eps 24–28** of the Logistic Regression series.*

Same five-line training loop as linear regression. Two substitutions: a sigmoid in the forward pass, binary cross-entropy in place of MSE. By the end you'll have trained a classifier three ways.

## Chapter 1 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## Chapter 2 · The dataset

Fake spam dataset. Two features per email:

- `x1` — number of suspicious words ("click here", "free", "urgent")
- `x2` — capitalization ratio (fraction of letters in CAPS)

Label is 1 if spam, 0 if not. Real spam classifiers use thousands of features. Two is enough to **see** what's going on.

In [ ]:
n_per_class = 60

not_spam = np.random.randn(n_per_class, 2) * 1.0 + np.array([1.5, 0.8])
spam     = np.random.randn(n_per_class, 2) * 1.0 + np.array([5.0, 4.0])

X = np.vstack([not_spam, spam])
y = np.hstack([np.zeros(n_per_class), np.ones(n_per_class)])

Sanity check.

In [ ]:
print('X.shape:', X.shape)
print('y.shape:', y.shape)
print('class balance:', int(y.sum()), 'spam  /  ', int((1-y).sum()), 'not-spam')

Plot it.

In [ ]:
plt.scatter(X[y==0, 0], X[y==0, 1], s=40, color='#00E5FF', label='not spam', edgecolor='black', linewidth=0.5)
plt.scatter(X[y==1, 0], X[y==1, 1], s=40, color='#FF4081', label='spam',     edgecolor='black', linewidth=0.5)
plt.xlabel('suspicious-word count'); plt.ylabel('capitalization ratio')
plt.title('Spam vs not-spam'); plt.legend(); plt.show()

Two clusters. Some overlap in the middle. **A line through that middle would separate most of them.** Finding that line is what we're going to do.

## Chapter 3 · Try linear regression first — watch it break

Labels are 0 and 1. Just numbers. Linear regression fits any numbers. Let's plug it in and see what happens.

In [ ]:
from sklearn.linear_model import LinearRegression

lr_bad = LinearRegression()
lr_bad.fit(X, y)

Predict for a few example emails.

In [ ]:
examples = np.array([[0.5, 0.0], [3.0, 2.0], [6.0, 5.5], [10.0, 8.0]])
for ex in examples:
    pred = lr_bad.predict(ex.reshape(1, -1))[0]
    print(f'  features {ex}  ->  predicted label {pred:+.3f}')

Look at those numbers.

- Email 1: prediction `-0.14`. *Negative.* What does negative probability mean? Nothing.
- Email 4: prediction `+2.06`. *More than 1.* Probability above 100%? Doesn't exist.

**Linear regression has no idea our output is supposed to be a probability.** We need a model that knows.

## Chapter 4 · The sigmoid function

The sigmoid takes any real number and squashes it into `(0, 1)`. Always. No matter how huge or tiny the input.

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

Worked example: what's `σ(0)`?

In [ ]:
sigmoid(0)

Exactly 0.5. The middle.

**Worked example:** what's `σ(1)`?

In [ ]:
sigmoid(1)

About 0.73. Mildly positive.

**Worked example:** what about a big positive input?

In [ ]:
sigmoid(5)

About 0.99. Confidently positive — but not 1.0.

**Worked example:** big negative input?

In [ ]:
sigmoid(-5)

About 0.0067. Confidently negative — but not 0.

**The asymptotes never reach.** Push the input to 100, the output gets really close to 1, but never exactly. That's the feature, not a bug — it keeps the math well-behaved.

Plot the full S-curve.

In [ ]:
zs = np.linspace(-10, 10, 200)
plt.plot(zs, sigmoid(zs), color='#FF4081', linewidth=2.5)
plt.axhline(0, color='gray', linewidth=0.5)
plt.axhline(1, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.xlabel('z'); plt.ylabel('σ(z)')
plt.title('The sigmoid — any real number into (0, 1)'); plt.show()

## Chapter 5 · Logistic regression = linear + sigmoid

Logistic regression is two steps:

1. **Linear part.** `z = b + w₁·x₁ + w₂·x₂ + …` Just like linear regression. Output is any real number.
2. **Sigmoid.** Squash `z` through the sigmoid. Output is in `(0, 1)`. That's the predicted probability.

$$\hat{y} = \sigma(b + w_1 x_1 + w_2 x_2)$$

Worked example. Bias `b=1`, weights `w₁=2, w₂=-1`. New email features `x₁=2, x₂=2`.

In [ ]:
b_eg = 1
w_eg = np.array([2, -1])
x_eg = np.array([2, 2])

z = b_eg + (w_eg * x_eg).sum()
print(f'  z = {z}')

Then squash.

In [ ]:
p = sigmoid(z)
print(f'  P(spam) = {p:.4f}')

About 95%. The model says: *given those features and those weights, this email has a 95% chance of being spam.*

Now — where do good weights come from? Training.

## Chapter 6 · Log loss

Before we train, we need a loss function. MSE doesn't work here — sigmoid's flat tails make the gradient vanish when the model is confidently wrong. (We'll see it in a moment.)

The right loss for binary classification is **log loss** (also called **binary cross-entropy**):

$$L = -\frac{1}{N}\sum_i \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

The formula has two cases packed into one. Let me unpack.

**Case 1: true label is 1.** The `(1 - yᵢ)` term is zero — drops out. Only `−log(ŷᵢ)` remains.

**Case 2: true label is 0.** The `yᵢ` term drops out. Only `−log(1 − ŷᵢ)` remains.

One formula, two cases. The case you're in depends on the truth.

In [ ]:
def log_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-9, 1 - 1e-9)   # avoid log(0)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

**Worked example.** Truth is 1, model predicts 0.92. Loss?

In [ ]:
log_loss(np.array([1]), np.array([0.92]))

Tiny. Model was right and confident.

**Worked example.** Truth is 1, model predicts 0.5 — uncertain.

In [ ]:
log_loss(np.array([1]), np.array([0.5]))

0.69. Moderate punishment for being uncertain.

**Worked example.** Truth is 1, model predicts 0.01 — confidently wrong.

In [ ]:
log_loss(np.array([1]), np.array([0.01]))

4.6. Big. **The loss explodes as you become more confidently wrong.** That's what we want — strong gradient signal exactly when the model needs to learn most.

Plot both curves.

In [ ]:
ps = np.linspace(0.001, 0.999, 200)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ps, -np.log(ps), color='#FF4081', linewidth=2.5)
axes[0].set_xlabel('predicted probability p'); axes[0].set_ylabel('loss')
axes[0].set_title('truth = 1   →   loss = -log(p)'); axes[0].set_ylim(0, 6)

axes[1].plot(ps, -np.log(1 - ps), color='#00E5FF', linewidth=2.5)
axes[1].set_xlabel('predicted probability p'); axes[1].set_ylabel('loss')
axes[1].set_title('truth = 0   →   loss = -log(1 - p)'); axes[1].set_ylim(0, 6)

plt.tight_layout(); plt.show()

Asymptote on the wrong side of each curve. Loss is small when the prediction matches the truth, blows up when it's confidently wrong.

## Chapter 7 · Train it by hand

Three parameters: `w₁`, `w₂`, `b`. Gradient descent is the same loop as linear regression, with sigmoid in the forward pass and log loss as the criterion.

Forward pass — compute predicted probabilities for the whole batch.

In [ ]:
def predict_prob(W, b, X):
    z = X @ W + b
    return sigmoid(z)

Gradient. (For log loss + sigmoid, the math is unusually clean — the gradients work out to `prediction − truth`, no chain-rule mess.)

In [ ]:
def grad_log(W, b, X, y):
    p = predict_prob(W, b, X)
    errs = p - y
    dW = (X.T @ errs) / len(y)
    db = np.mean(errs)
    return dW, db

Initialize parameters.

In [ ]:
W = np.zeros(2)
b = 0.0
lr = 0.1

One step.

In [ ]:
dW, db = grad_log(W, b, X, y)
print(f'before:  W={W}  b={b:.3f}  loss={log_loss(y, predict_prob(W, b, X)):.4f}')

W -= lr * dW
b -= lr * db

print(f'after:   W={W.round(3)}  b={b:.3f}  loss={log_loss(y, predict_prob(W, b, X)):.4f}')

Loss went down. Now run the full loop.

We run **10,000 iterations** — a lot for three parameters. Same reason as the linear-regression notebook: the features aren't normalized (word counts run 0–10, caps ratio 0–8), so plain gradient descent takes many small steps to get there.

In [ ]:
W = np.zeros(2)
b = 0.0
history = []

for step in range(10000):
    dW, db = grad_log(W, b, X, y)
    W -= lr * dW
    b -= lr * db
    history.append(log_loss(y, predict_prob(W, b, X)))

print(f'final W = {W.round(3)}   b = {b:.3f}   log loss = {history[-1]:.4f}')

Plot the loss curve.

In [ ]:
plt.plot(history, color='#FF4081', linewidth=2)
plt.xlabel('step'); plt.ylabel('log loss')
plt.title('Training — loss going down over 10,000 iterations')
plt.show()

## Chapter 8 · The decision boundary

Plot the trained model's *decision boundary*. For logistic regression, the boundary is the line where predicted probability equals 0.5 — equivalently, where `W·x + b = 0`.

In [ ]:
# Boundary: w0*x0 + w1*x1 + b = 0  →  x1 = -(w0*x0 + b) / w1
x0_range = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)
boundary_x1 = -(W[0] * x0_range + b) / W[1]

plt.scatter(X[y==0, 0], X[y==0, 1], s=40, color='#00E5FF', label='not spam', edgecolor='black', linewidth=0.5)
plt.scatter(X[y==1, 0], X[y==1, 1], s=40, color='#FF4081', label='spam',     edgecolor='black', linewidth=0.5)
plt.plot(x0_range, boundary_x1, color='#FFD27F', linewidth=2.5, label='decision boundary')
plt.xlabel('suspicious-word count'); plt.ylabel('capitalization ratio')
plt.title('Trained logistic regression — boundary at p = 0.5'); plt.legend(); plt.show()

Yellow line splits spam from not-spam. On this dataset the two clusters barely overlap, so the trained boundary separates them cleanly — every point on its own side. Real inboxes are messier: spam and ham blur together, points land on the wrong side, and *where* you draw the threshold (next series, classification) is the precision/recall trade-off.

## Chapter 9 · The same thing in PyTorch — 5-line loop, two changes from LR

Same five-line loop from the linear regression notebook. Two substitutions:

- Add a sigmoid in the forward pass.
- Use binary cross-entropy as the loss.

PyTorch's `nn.BCEWithLogitsLoss` combines sigmoid + BCE in one call (more numerically stable).

In [ ]:
import torch
import torch.nn as nn

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32).view(-1, 1)

Set up model + optimizer + loss.

In [ ]:
model = nn.Linear(2, 1)        # 2 features in, 1 logit out
nn.init.zeros_(model.weight); nn.init.zeros_(model.bias)   # start from zeros, like the by-hand run
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.BCEWithLogitsLoss()

The five lines.

In [ ]:
for step in range(10000):
    logits = model(X_t)        # forward — no sigmoid here; BCEWithLogitsLoss adds it
    loss = loss_fn(logits, y_t)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

Compare to the by-hand version.

In [ ]:
W_torch = model.weight.detach().numpy().flatten()
b_torch = model.bias.item()

print(f'  PyTorch:   W={W_torch.round(3)}   b={b_torch:.3f}')
print(f'  By-hand:   W={W.round(3)}   b={b:.3f}')

Same answer. **Three lines of the five-line loop are literally identical to linear regression.** Two changes give you a classifier.

## Chapter 10 · The same thing in sklearn — 3 lines

In [ ]:
from sklearn.linear_model import LogisticRegression

model_sk = LogisticRegression()
model_sk.fit(X, y)

Read out the coefficients.

In [ ]:
print(f'  sklearn coefficients:  {model_sk.coef_[0].round(3)}')
print(f'  sklearn intercept:     {model_sk.intercept_[0]:.3f}')

Sklearn's coefficients differ from ours because **sklearn applies mild L2 regularization by default** — it keeps the weights from drifting too large. Our by-hand fit has none, and because these two clusters are nearly separable, its weights keep creeping upward the longer it trains. So ours land a bit *larger* than sklearn's here. Same boundary direction either way, and the predictions agree.

Predict probabilities for a few example emails.

In [ ]:
for ex in examples:
    probs = model_sk.predict_proba(ex.reshape(1, -1))[0]
    print(f'  features {ex}  ->  P(not spam)={probs[0]:.3f}  P(spam)={probs[1]:.3f}')

Real probabilities now. All between 0 and 1. **That's what we built sigmoid for.**

## Chapter 11 · Bonus — sklearn's algorithm roadmap

Sklearn ships with an [infamous algorithm cheat sheet](https://scikit-learn.org/stable/machine_learning_map.html) — answer a few questions about your data, it routes you to a starting model.

![sklearn algorithm map](https://scikit-learn.org/stable/_static/ml_map.svg)

**Walking the chart for our spam problem:**

1. 50+ samples? **Yes** (120 emails).
2. Predicting a category? **Yes** → classification branch.
3. Labeled data? **Yes** → supervised classification.
4. <100K samples? **Yes** → small-classifier zone: `LinearSVC`, `KNeighborsClassifier`, `SVC`.

We used `LogisticRegression` — which sits next door in sklearn's mental map. The chart's default for small-data classification is `LinearSVC` (a margin-based classifier), but `LogisticRegression` is what most people actually reach for first because:

- It outputs **probabilities**, not just labels. (We need these in the classification series — threshold, ROC, precision/recall all start from probabilities.)
- The decision boundary and coefficients are **interpretable** — you can read what the model learned.
- It's the **simplest classifier** that has a real probabilistic interpretation.

**The chart isn't gospel.** It's a starting point. The workflow:

1. Pick a baseline from the chart.
2. Train it.
3. Look at the errors.
4. Iterate to a better model (random forest, gradient boosting, neural net…).

Logistic regression rarely wins competitions. But it's almost always the first thing to try — fast, interpretable, well-understood. If your fancy 50M-parameter neural net only marginally beats logistic regression, that's a sign your problem doesn't need 50M parameters.

## Chapter 12 · Read the trained model

Coefficients in logistic regression have a clean interpretation — they're effects on the **log-odds**. Exponentiating a coefficient gives the multiplicative effect on the odds.

In [ ]:
coef_word, coef_caps = model_sk.coef_[0]
intercept = model_sk.intercept_[0]

print('In log-odds terms:')
print(f'  each suspicious word adds {coef_word:+.2f} to the log-odds of spam')
print(f'  each unit of caps ratio adds {coef_caps:+.2f}')
print(f'  baseline log-odds (no features): {intercept:+.2f}')

Exponentiate to get the multiplicative effects.

In [ ]:
print('In odds-multiplier terms:')
print(f'  one more suspicious word multiplies the odds of spam by  {np.exp(coef_word):.2f}x')
print(f'  one unit more caps ratio multiplies the odds by          {np.exp(coef_caps):.2f}x')

**That's a real model you can interpret.** Each feature has a clear, multiplicative effect on the odds of spam. Not a black box.

## Chapter 13 · What we did

1. Built a synthetic spam dataset (two features, two classes).
2. Tried linear regression — saw it produce probabilities below 0 and above 1. Broken.
3. Introduced the sigmoid function. Worked through four numeric examples to feel it.
4. Combined linear + sigmoid into logistic regression. Worked example.
5. Introduced log loss. Three worked examples to feel why it's the right loss.
6. Trained the model by hand. One step, then the full loop.
7. Plotted the decision boundary.
8. Same model in PyTorch (5 lines, two substitutions from LR) and sklearn (3 lines).
9. Read the trained coefficients as multiplicative effects on the odds.

**Next series: classification.** Threshold. Confusion matrix. Precision, recall, ROC. How to know whether your classifier is actually good.